<a href="https://colab.research.google.com/github/NeoByteVoyager/PytorchLearning/blob/main/ML/Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 一.Python基础

## 1.张量基础

In [ ]:
import torch
# 创建张量
t=torch.tensor([[1,2,3],[4,5,6]])
# 打印张量
print(t)
# 查看张量形状
print(t.shape)
# 查看类型
print(t.dtype)



tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])
torch.int64


## 2.基本运算


In [ ]:
a=torch.tensor([1.0,2.0,3.0])
b=torch.tensor([4.0,5.0,6.0])
# 加法
print(a+b)
print(torch.add(a,b))

# 逐个元素相乘
print(a*b)

# 矩阵相乘
mat_a=torch.tensor([[1,2],[3,4]])
mat_b=torch.tensor([[5,6],[7,8]])
print(torch.matmul(mat_a,mat_b))


tensor([5., 7., 9.])
tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
tensor([[19, 22],
        [43, 50]])


## 3.改变形状

In [ ]:
x=torch.arange(12)
print(x)
x=x.reshape(3,4)
print(x)
print(x.view(2,6))
print(x)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


# 二.自动求导

In [ ]:
x=torch.tensor([2.,3.],requires_grad=True)
y=2*x
print(y)
# y.backward()  错误示例，y也是一个张量，但是backward的对象应该是一个标量只有一个数值
loss=y.sum()*2
loss.backward()
print(x.grad)

tensor([4., 6.], grad_fn=<MulBackward0>)
tensor([4., 4.])


## 三.线性模型构建


对样本内存存储的理解：
- 为了让存储空间连续，一个样本的是一行，即特征向量是按行向量进行存储的   

Linear的理解：
- 第一个参数是输入向量的特征维度(in_feature)
- 第二个参数是输出向量的特征维度(out_feature)
- 实际上Linear创建一个(in_feature,out_feature)的矩阵W
- 实际上前向传播的过程是x和W相乘的过程

In [ ]:
import torch
import torch.nn as nn
# 训练数据
X=torch.tensor([[52.0,54.0,56.0],
        [2.0,4.0,6.0],
        [1.2,1.4,1.6],
        [5.4,5.6,5.8],
        [3.0,6.0,9.0]],dtype=torch.float32)
Y=torch.tensor([[54.0],[4.0],[1.4],[5.6],[6.0]],dtype=torch.float32)
# 构建一层模型
model=nn.Linear(3,1)
# 定义损失函数（MSE）和优化器(SGD)
criterion=nn.MSELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)
# 训练循环
for epoch in range(300):
  optimizer.zero_grad() #清空残留梯度
  # 前向传播
  Y_hat=model(X)
  loss=criterion(Y,Y_hat)
  # 反向传播
  loss.backward()
  print(f"第{epoch}轮的loss:{loss.item()},梯度：{model.weight}，Bias:{model.bias}")
  # 更新参数
  optimizer.step()

## 四.手搓线性模型


**注意点:**
  - 手搓模型继承了nn.Module
  - 在进行初始化是要先**对基类进行初始化**
  - 参数设置为Parameter形式，tensor在套上它之后`require_grad=True`,反向传播时会计算它的梯度并更新它
  - 在基类中定义了`__call__`，相当于`()`重载运算符。我们可以直接写`model(x)`实现前向传播。不能显式调用`model.forward()`，`__call__`内部还写明了自动求导的逻辑,直接调用`forward()`不能自动求导和反向传播


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
class Mymodel(nn.Module):
  def __init__(self,in_features,out_features):
    super().__init__()

    self.weight=nn.Parameter(torch.randn(out_features,in_features))
    self.bias=nn.Parameter(torch.randn(out_features))
  def forward(self,x):
    return x @ self.weight.t()+self.bias
# 训练数据
X=torch.tensor([[52.0,54.0,56.0],
        [2.0,4.0,6.0],
        [1.2,1.4,1.6],
        [5.4,5.6,5.8],
        [3.0,6.0,9.0]],dtype=torch.float32)
Y=torch.tensor([[54.0],[4.0],[1.4],[5.6],[6.0]],dtype=torch.float32)
# 构建一层模型
model=Mymodel(3,1)
# 定义损失函数（MSE）和优化器(SGD)
criterion=nn.MSELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)
# 训练循环
for epoch in range(300):
  optimizer.zero_grad() #清空残留梯度
  # 前向传播
  Y_hat=model(X)
  loss=criterion(Y,Y_hat)
  # 反向传播
  loss.backward()
  print(f"第{epoch}轮的loss:{loss.item()},梯度：{model.weight}，Bias:{model.bias}")
  # 更新参数
  optimizer.step()

## 五.手搓深度学习

深度学习就是**多层线性模型的叠加**，**每一层模型的输出对应下一层模型的输入**，同时使用**非线性函数**，使其具备模拟复杂情况的能力

In [ ]:
class MyThreeLayerNet(nn.Module):
  def __init__(self,input_dim,hidden_dim1,hidden_dim2,output_dim):
    super().__init__()
    self.layer1=nn.Linear(input_dim,hidden_dim1)
    self.layer2=nn.Linear(hidden_dim1,hidden_dim2)
    self.layer3=nn.Linear(hidden_dim2,output_dim)

    self.relu=nn.ReLU()
  def forward(self,x):
    # 第一层
    x=self.layer1(x)
    x=self.relu(x)
    # 第二层
    x=self.layer2(x)
    x=self.relu(x)
    # 第三层
    output=self.layer3(x)

    return output
# 训练数据
X=torch.tensor([[2.0,4.0,6.0],
        [1.2,1.4,1.6],
        [5.4,5.6,5.8],
        [3.0,6.0,9.0]],dtype=torch.float32)
Y=torch.tensor([[4.0],[1.4],[5.6],[6.0]],dtype=torch.float32)
# 构建一层模型
model=MyThreeLayerNet(3,5,3,1)
# 定义损失函数（MSE）和优化器(SGD)
criterion=nn.MSELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)
loss_history=[]
# 训练循环
for epoch in range(300):
  optimizer.zero_grad() #清空残留梯度
  # 前向传播
  Y_hat=model(X)
  loss=criterion(Y,Y_hat)

  # 反向传播
  loss.backward()
  # 每 50 轮打印一次，方便观察
  if epoch % 50 == 0:
      print(f"\n--- Epoch {epoch} ---")
      print(f"当前 Loss: {loss.item():.6f}")

      print(f"Layer1 权重:\n {model.layer1.weight.data}")
      print(f"Layer1 梯度:\n {model.layer1.weight.grad.data}")
      print(f"Layer1 偏置: {model.layer1.bias.data}")

      print(f"Layer2 权重:\n {model.layer2.weight.data}")
      print(f"Layer2 梯度:\n {model.layer2.weight.grad.data}")
      print(f"Layer2 偏置: {model.layer2.bias.data}")

      print(f"Layer3 权重:\n {model.layer3.weight.data}")
      print(f"Layer3 梯度:\n {model.layer3.weight.grad.data}")
      print(f"Layer3 偏置: {model.layer3.bias.data}")
      loss_history.append(loss.item())
  # 更新参数
  optimizer.step()
print(loss_history)



**item(),data,tensor的区别：**

|类型|是否带计算图|得到的结果|应用场景|
|---|---|---|---|
|tensor.item()|否|原生数字|记录/打印**标量**，如loss|
|tensor.data|否|纯净的张量|打印/查看多维权重|
|tensor|是|完整的矿建|在forward中做数学运算(要能反向传播)|